# Priors

A GPflow 2 prior example using TensorFlow Probability distributions on DGP parameters.
The model objective is still the DGP ELBO; the prior terms can be added by a training loop when MAP-style regularization is desired.


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
import gpflow
from gpflow.likelihoods import Gaussian

from doubly_stochastic_dgp.dgp import DGP

gpflow.config.set_default_float(np.float64)
tfd = tfp.distributions

X = np.linspace(-1.0, 1.0, 12)[:, None]
Y = np.sin(3.0 * X)
Z = X[::3].copy()
kernels = [gpflow.kernels.SquaredExponential(), gpflow.kernels.SquaredExponential()]
model = DGP(X, Y, Z, kernels, Gaussian(), num_samples=2)

for layer in model.layers:
    layer.kern.variance.prior = tfd.LogNormal(tf.cast(-1.0, gpflow.default_float()), tf.cast(0.5, gpflow.default_float()))
    layer.kern.lengthscales.prior = tfd.LogNormal(tf.cast(0.0, gpflow.default_float()), tf.cast(0.5, gpflow.default_float()))
model.likelihood.likelihood.variance.prior = tfd.LogNormal(tf.cast(-2.0, gpflow.default_float()), tf.cast(0.5, gpflow.default_float()))

parameters_with_priors = [
    model.layers[0].kern.variance,
    model.layers[0].kern.lengthscales,
    model.layers[1].kern.variance,
    model.layers[1].kern.lengthscales,
    model.likelihood.likelihood.variance,
]
log_prior = tf.add_n([tf.reduce_sum(parameter.prior.log_prob(parameter)) for parameter in parameters_with_priors])
elbo = model.maximum_log_likelihood_objective()

float(elbo.numpy()), float(log_prior.numpy())
